In [1]:
using ITensors, ITensorMPS
using ITransverse
using ITensorExpMPO
using ITensors: Algorithm
using Plots
using Plots.PlotMeasures
using ProgressMeter
ProgressMeter.ijulia_behavior(:clear)
using JLD2


using LsqFit

using Revise
includet("main.jl")

In [2]:
# Global Parameters
N = 40
target_times = collect(0.5 : 1.0 : 6.5) 
p = 0.5
lambda = 1.0
dt = 0.05
cutoff = 1e-14
maxdim = 256

# Setup finite chain and shared initial state
sites = siteinds("S=1/2", N)
psi0 = complex(MPS(sites, "X+"))

40-element MPS:
 ((dim=2|id=983|"S=1/2,Site,n=1"), (dim=1|id=642|"Link,l=1"))
 ((dim=1|id=642|"Link,l=1"), (dim=2|id=550|"S=1/2,Site,n=2"), (dim=1|id=340|"Link,l=2"))
 ((dim=1|id=340|"Link,l=2"), (dim=2|id=945|"S=1/2,Site,n=3"), (dim=1|id=108|"Link,l=3"))
 ((dim=1|id=108|"Link,l=3"), (dim=2|id=769|"S=1/2,Site,n=4"), (dim=1|id=425|"Link,l=4"))
 ((dim=1|id=425|"Link,l=4"), (dim=2|id=746|"S=1/2,Site,n=5"), (dim=1|id=238|"Link,l=5"))
 ((dim=1|id=238|"Link,l=5"), (dim=2|id=74|"S=1/2,Site,n=6"), (dim=1|id=625|"Link,l=6"))
 ((dim=1|id=625|"Link,l=6"), (dim=2|id=775|"S=1/2,Site,n=7"), (dim=1|id=396|"Link,l=7"))
 ((dim=1|id=396|"Link,l=7"), (dim=2|id=710|"S=1/2,Site,n=8"), (dim=1|id=327|"Link,l=8"))
 ((dim=1|id=327|"Link,l=8"), (dim=2|id=734|"S=1/2,Site,n=9"), (dim=1|id=346|"Link,l=9"))
 ((dim=1|id=346|"Link,l=9"), (dim=2|id=262|"S=1/2,Site,n=10"), (dim=1|id=512|"Link,l=10"))
 ⋮
 ((dim=1|id=594|"Link,l=31"), (dim=2|id=533|"S=1/2,Site,n=32"), (dim=1|id=402|"Link,l=32"))
 ((dim=1|id=402|"Link,l=3

In [ ]:
# TVDP 
H_mpo = MPO(benchmark_opsum(N, lambda, p), sites)
psi_t_TDVP = deepcopy(psi0)
rate_TDVP = Float64[]
current_t = 0.0

@showprogress "TDVP Evolution..." for T in target_times
    steps = round(Int, (T - current_t) / dt)
    for _ in 1:steps
        global psi_t_TDVP = tdvp(H_mpo, -im * dt, psi_t_TDVP; cutoff=cutoff, maxdim=maxdim, nsite=2)
        normalize!(psi_t_TDVP) 
    end
    global current_t = T
    
    G_TDVP = inner(psi0, psi_t_TDVP)
    push!(rate_TDVP, -log(max(abs(G_TDVP), 1e-50)) / N)
    GC.gc()
end

jldsave("rate_TDVP.jld2"; rate_TDVP)

psi_t_TDVP = nothing
H_mpo = nothing
GC.gc()

In [ ]:
# VD2 
U_VD2 = expH_benchmark(sites, lambda, p; dt=dt, mpo_alg="VD2")
psi_t_VD2 = deepcopy(psi0)
rate_VD2 = Float64[]
current_t = 0.0

@showprogress "VD2 Evolution..." for T in target_times
    steps = round(Int, (T - current_t) / dt)
    for _ in 1:steps
        global psi_t_VD2 = apply(U_VD2, psi_t_VD2; cutoff=cutoff, maxdim=maxdim)
        normalize!(psi_t_VD2) 
    end
    global current_t = T
    
    G_VD2 = inner(psi0, psi_t_VD2)
    push!(rate_VD2, -log(max(abs(G_VD2), 1e-50)) / N)
    GC.gc()
end

jldsave("rate_VD2.jld2"; rate_VD2)

psi_t_VD2 = nothing
U_VD2 = nothing
GC.gc()

In [ ]:
# WII
U_WII = expH_benchmark(sites, lambda, p; dt=dt, mpo_alg="WII")
psi_t_WII = deepcopy(psi0)
rate_WII = Float64[]
current_t = 0.0

@showprogress "WII Evolution..." for T in target_times
    steps = round(Int, (T - current_t) / dt)
    for _ in 1:steps
        global psi_t_WII = apply(U_WII, psi_t_WII; cutoff=cutoff, maxdim=maxdim)
        normalize!(psi_t_WII) 
    end
    global current_t = T
    
    G_WII = inner(psi0, psi_t_WII)
    push!(rate_WII, -log(max(abs(G_WII), 1e-50)) / N)
    GC.gc()
end

jldsave("rate_WII.jld2"; rate_WII)

psi_t_WII = nothing
U_WII = nothing
GC.gc()

In [ ]:
target_times = collect(0.5 : 1.0 : 6.5) 

# Load data safely from the JLD2 files
r_TDVP = load("rate_TDVP.jld2", "rate_TDVP")
r_VD2  = load("rate_VD2.jld2", "rate_VD2")
r_WII  = load("rate_WII.jld2", "rate_WII")

comparison_plot = plot(
    title="Benchmark Loschmidt Echo: TDVP vs VD2 vs WII",
    xlabel="Time (t)", 
    ylabel="Rate Function l(t)", 
    grid=true, framestyle=:box, legend=:topleft
)

plot!(comparison_plot, target_times, r_TDVP, label="TDVP (nsite=2)", lw=3, color=:black)
plot!(comparison_plot, target_times, r_VD2, label="VD2 MPO", lw=2, marker=:circle, ls=:dash, color=:blue)
plot!(comparison_plot, target_times, r_WII, label="WII MPO", lw=2, marker=:cross, ls=:dot, color=:red)